[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/twobombs/thereminq-examples/blob/main/devices/stellarator/stellarator_colab.ipynb)

# W7-X Kitaev Stellarator: live dashboard (Colab / Jupyter)

Notebook version of `stellarator_ui.py`: a W7-X-like stellarator built from Kitaev Z2 lattice-gauge parts (1080 sites, 40 x 27-qubit manifolds), simulated with an exact Majorana reference and PyQrack. Pick a job, set parameters and sweep ranges, press **Run**, and watch every step arrive in the charts and the 3D view.

**Run it:** *Runtime > Run all* (a standard CPU runtime is enough). Cell 1 installs `pyqrack-cpu` (or OpenCL `pyqrack` on GPU runtimes with OpenCL), `plotly>=6`, and `anywidget` (preserving Colab's built-in `ipywidgets`) and fetches the four files it needs (`stellarator.py`, `stellarator_ui.py`, `manifest.json`, `scoreboard.json`) from `devices/stellarator` on GitHub, or lets you upload them.

| Job | What it computes | Speed on a Colab CPU |
| :--- | :--- | :--- |
| NESS pulse | Core-to-divertor Lindblad steady state on the 1080-site machine | about 0.3 to 0.4 s per step |
| Coil landscape | 3D Biot-Savart vacuum metrics for the 7 coil knobs (X / Y heatmap) | about 2 ms per point |
| 7 criteria | Full W7-X 7-criteria evaluation | about 60 s per point |
| PyQrack manifold | 16q / 27q block flow vs the exact Majorana propagator | 16q about 0.06 s, 27q about 5 s per step |
| Ebit link budget | DEJMPS distilled-ebit cost vs LO-QPD shot count | instant |


In [ ]:
#@title 1. Install packages and fetch the stellarator files
SOURCE = "github"  #@param ["github", "upload"]
QRACK_BACKEND = "auto"  #@param ["auto", "cpu", "opencl"]
REPO = "https://github.com/twobombs/thereminq-examples.git"
BRANCH = "main"
SUBDIR = "devices/stellarator"

import ctypes
import glob
import importlib.metadata
import io
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.request

IN_COLAB = "google.colab" in sys.modules
PIP = [sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore"]


def _run(cmd, check=True):
    return subprocess.run(cmd, check=check)


def _apt_install(pkgs):
    if not pkgs or not shutil.which("apt-get") or os.geteuid() != 0:
        return False
    _run(["apt-get", "update", "-qq"], check=False)
    res = _run(["apt-get", "install", "-y", "-qq"] + list(pkgs), check=False)
    return res.returncode == 0


def _has_lib(name):
    try:
        ctypes.CDLL(name)
        return True
    except OSError:
        return False


def _test_pyqrack():
    code = (
        "from pyqrack import QrackSimulator; "
        "s = QrackSimulator(2, is_gpu=False); s.h(0); s.mcx([0], 1); "
        "assert abs(s.prob(0) - 0.5) < 1e-5"
    )
    return subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)


# 1. Ensure system BLAS (needed by both pyqrack and pyqrack-cpu on Linux)
if sys.platform.startswith("linux") and not _has_lib("libopenblas.so.0"):
    _apt_install(["libopenblas0"])

# 2. Install PyQrack (default to pyqrack-cpu on CPU runtimes to avoid missing libclblast/OpenCL)
want_opencl = QRACK_BACKEND == "opencl"
if QRACK_BACKEND == "auto" and sys.platform.startswith("linux"):
    has_gpu = bool(shutil.which("nvidia-smi")) and subprocess.run(
        ["nvidia-smi", "-L"], capture_output=True
    ).returncode == 0
    has_icd = bool(glob.glob("/etc/OpenCL/vendors/*.icd"))
    want_opencl = has_gpu and has_icd

if _test_pyqrack().returncode != 0 or QRACK_BACKEND in ("cpu", "opencl"):
    if want_opencl:
        if sys.platform.startswith("linux"):
            missing = []
            if not _has_lib("libopenblas.so.0"):
                missing.append("libopenblas0")
            if not _has_lib("libclblast.so.1"):
                missing.append("libclblast1")
            if not _has_lib("libOpenCL.so.1"):
                missing.append("ocl-icd-libopencl1")
            _apt_install(missing)
        _run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "pyqrack-cpu"], check=False)
        _run(PIP + ["pyqrack"], check=False)
        if _test_pyqrack().returncode != 0:
            print("OpenCL pyqrack check failed; falling back to pyqrack-cpu.")
            _run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "pyqrack"], check=False)
            _run(PIP + ["pyqrack-cpu"], check=True)
    else:
        _run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "pyqrack"], check=False)
        _run(PIP + ["pyqrack-cpu"], check=True)

probe = _test_pyqrack()
if probe.returncode != 0:
    raise RuntimeError("PyQrack failed to initialize:\n" + probe.stderr)

# 3. Install plotting/widget stack (preserve Colab's ipywidgets 7.x frontend compatibility)
pkgs = ["plotly>=6", "anywidget"]
if IN_COLAB:
    try:
        iw_ver = importlib.metadata.version("ipywidgets")
        if int(iw_ver.split(".")[0]) < 8:
            pkgs.append("ipywidgets==" + iw_ver)
    except importlib.metadata.PackageNotFoundError:
        pkgs.append("ipywidgets>=7.7,<8")
else:
    pkgs.append("ipywidgets>=8")
_run(PIP + pkgs, check=True)

# 4. Fetch the four required stellarator files
NEEDED = ["stellarator.py", "stellarator_ui.py", "manifest.json", "scoreboard.json"]


def have(d):
    return all(os.path.exists(os.path.join(d, f)) for f in NEEDED)


def extract_missing_from_tar(tar_path, dst):
    with tarfile.open(tar_path, mode="r:gz") as tf:
        for mem in tf.getmembers():
            base = os.path.basename(mem.name)
            out_path = os.path.join(dst, base)
            if mem.isfile() and base in NEEDED and not os.path.exists(out_path):
                with open(out_path, "wb") as fh:
                    fh.write(tf.extractfile(mem).read())


WORK = "/content/stellarator" if IN_COLAB else os.path.abspath("stellarator_work")
if have(os.getcwd()):                      # notebook opened inside the repo folder
    WORK = os.getcwd()
elif not have(WORK):
    os.makedirs(WORK, exist_ok=True)
    if SOURCE == "github":
        tmp = "/tmp/thereminq-src"
        shutil.rmtree(tmp, ignore_errors=True)
        try:
            _run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                  "-b", BRANCH, REPO, tmp], check=True)
            _run(["git", "-C", tmp, "sparse-checkout", "set", SUBDIR], check=True)
            src = os.path.join(tmp, SUBDIR)
            for fname in NEEDED:
                fpath = os.path.join(src, fname)
                if os.path.exists(fpath):
                    shutil.copy2(fpath, os.path.join(WORK, fname))
            if not have(WORK):
                for tgz in sorted(glob.glob(os.path.join(src, "*.tar.gz"))):
                    extract_missing_from_tar(tgz, WORK)
        except Exception as exc:
            print("Git fetch failed (%s); trying direct GitHub raw download..." % exc)
        if not have(WORK) and "github.com/" in REPO:
            slug = REPO.split("github.com/", 1)[1].removesuffix(".git").strip("/")
            raw_base = "https://raw.githubusercontent.com/%s/%s/%s" % (slug, BRANCH, SUBDIR.strip("/"))
            for fname in NEEDED:
                dst = os.path.join(WORK, fname)
                if not os.path.exists(dst):
                    try:
                        urllib.request.urlretrieve("%s/%s" % (raw_base, fname), dst)
                    except Exception:
                        pass
        if not have(WORK):
            print("The dashboard files are not on GitHub (%s/%s) yet; falling back to upload." % (BRANCH, SUBDIR))
    if not have(WORK):
        if not IN_COLAB:
            raise SystemExit("Put %s next to this notebook, or into %s" % (", ".join(NEEDED), WORK))
        from google.colab import files
        print("Upload the four files, or the stellarator-*.tar.gz archive:")
        for name, data in files.upload().items():
            if name.endswith((".tar.gz", ".tgz")):
                with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tf:
                    for mem in tf.getmembers():
                        base = os.path.basename(mem.name)
                        if mem.isfile() and base in NEEDED:
                            with open(os.path.join(WORK, base), "wb") as fh:
                                fh.write(tf.extractfile(mem).read())
            else:
                with open(os.path.join(WORK, os.path.basename(name)), "wb") as fh:
                    fh.write(data)

assert have(WORK), "missing files in %s: %s" % (WORK, [f for f in NEEDED if not os.path.exists(os.path.join(WORK, f))])
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

import anywidget, ipywidgets, plotly
print("Working directory:", WORK)
print("Stack ready: pyqrack OK | plotly %s | anywidget %s | ipywidgets %s"
      % (plotly.__version__, anywidget.__version__, ipywidgets.__version__))


In [ ]:
#@title 2. Load the physics engine and build the machine
import os, sys, time

IN_COLAB = "google.colab" in sys.modules
QRACK_LIB_PATH = "/usr/local/lib/qrack/libqrack_pinvoke.so"
if os.path.exists(QRACK_LIB_PATH):
    os.environ.setdefault("PYQRACK_SHARED_LIB_PATH", QRACK_LIB_PATH)

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()      # needed for live plotly FigureWidgets

if "WORK" in globals() and WORK not in sys.path:
    sys.path.insert(0, WORK)

import stellarator_ui as U
t0 = time.time()
machine = U.LiveMachine("manifest.json", "scoreboard.json")
ctx = machine.ctx()
runner = U.LiveJobRunner(machine)
print("Machine ready: %d sites, %d bonds, %d seam bonds (%.1f s)"
      % (ctx["n_sites"], len(ctx["bonds"]), len(ctx["seam_bonds"]), time.time() - t0))


In [ ]:
#@title 3. Dashboard code
import contextlib
import csv
import json
import math
import sys
import threading
import time

import numpy as np
import ipywidgets as W
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import stellarator_ui as U

C = U.UI_C
LAB, FMT, DUR = U._ui_lab, U._ui_fmt, U._ui_dur
DARK = dict(paper_bgcolor=C["surf"], plot_bgcolor=C["surf"],
            font=dict(color=C["t2"], size=11, family="system-ui, -apple-system, Segoe UI, Roboto, sans-serif"),
            margin=dict(l=60, r=30, t=10, b=46))
AX = dict(visible=True, gridcolor=C["grid"], zerolinecolor=C["grid"], linecolor=C["axis"], tickcolor=C["axis"],
          tickfont=dict(color=C["t3"]), title_font=dict(color=C["t2"]))


def _scale(ramp):
    return [[k / (len(ramp) - 1), c] for k, c in enumerate(ramp)]


CS_SEQ = _scale(U.UI_SEQ)
CS_ORANGE = _scale(U.UI_ORANGE)
CS_DIV = _scale(U.UI_DIV_NEG + [C["mid"]] + U.UI_DIV_POS[::-1])
CS_TRACE = U.UI_SEQ[2:]
# 3D view: drop the near-white far-negative step so blue and red extremes carry equal weight on the dark scene
CS_DIV3D = _scale(U.UI_DIV_NEG[1:] + [C["mid"]] + U.UI_DIV_POS[1:][::-1])
BG3D = "#161615"


def _dim(hexcol, a):
    """Blend a colour into the 3D scene background. Used instead of trace opacity: WebGL
    transparency passes inside notebook widget frames can composite colours inverted."""
    r, g, b = U._ui_lerp_hex([BG3D, hexcol], a)
    return "#%02x%02x%02x" % (int(round(r)), int(round(g)), int(round(b)))


CS_DIV3D_DIM = [[t, _dim(c, 0.45)] for t, c in CS_DIV3D]


def _rgb(t, ramp):
    r, g, b = U._ui_lerp_hex(ramp, t)
    return "rgb(%d,%d,%d)" % (r, g, b)


def _segments(P, bonds, ids):
    """x, y, z arrays for line segments with NaN gaps (plotly draws them as breaks)."""
    ids = np.asarray(ids, dtype=int)
    if not len(ids):
        return [], [], []
    a = P[[bonds[k][0] for k in ids]]
    b = P[[bonds[k][1] for k in ids]]
    gap = np.full((len(ids), 3), np.nan)
    seg = np.stack([a, b, gap], axis=1).reshape(-1, 3)
    return seg[:, 0], seg[:, 1], seg[:, 2]


class ParamForm(object):
    """ipywidgets form for one job: value editors plus optional sweep ranges."""

    def __init__(self, job, spec, on_change):
        self.job, self.spec = job, spec
        self.ed, self.sx, self.sy, self.rng = {}, {}, {}, {}
        rows = []
        has_coils = any(q["key"] == "coils" for q in spec["params"])
        for pd in spec["params"]:
            name = pd["label"] + (" (custom coils)" if pd.get("group") == "coils" and has_coils else "")
            lab = W.HTML("<span style='color:%s'>%s</span>" % (C["t2"], name), layout=W.Layout(width="170px"))
            if pd["type"] == "select":
                ed = W.Dropdown(options=pd["options"], value=str(pd["default"]), layout=W.Layout(width="150px"))
            elif pd["type"] == "int":
                ed = W.BoundedIntText(value=int(round(float(pd["default"]))), min=int(pd["min"]), max=int(pd["max"]),
                                      layout=W.Layout(width="110px"))
            else:
                ed = W.BoundedFloatText(value=float(pd["default"]), min=float(pd["min"]), max=float(pd["max"]),
                                        step=None, layout=W.Layout(width="110px"))
            ed.observe(lambda ch: on_change(), names="value")
            self.ed[pd["key"]] = ed
            row = [lab, ed]
            if pd["type"] != "select":
                cx = W.Checkbox(value=False, description="sweep", indent=False, layout=W.Layout(width="70px"))
                cx.observe(lambda ch, k=pd["key"]: self._toggle(k, "x", ch["new"], on_change), names="value")
                self.sx[pd["key"]] = cx
                row.append(cx)
                if spec.get("two_d"):
                    cy = W.Checkbox(value=False, description="Y", indent=False, layout=W.Layout(width="44px"))
                    cy.observe(lambda ch, k=pd["key"]: self._toggle(k, "y", ch["new"], on_change), names="value")
                    self.sy[pd["key"]] = cy
                    row.append(cy)
            rows.append(W.HBox(row, layout=W.Layout(align_items="center")))
            if pd["type"] != "select":
                lo, hi = float(pd["min"]), float(pd["max"])
                if not pd.get("log"):
                    lo, hi = lo + 0.15 * (hi - lo), hi - 0.15 * (hi - lo)
                mk = (lambda v: W.BoundedIntText(value=int(round(v)), min=int(pd["min"]), max=int(pd["max"]), layout=W.Layout(width="96px"))
                      if pd["type"] == "int" else
                      W.BoundedFloatText(value=float("%.4g" % v), min=float(pd["min"]), max=float(pd["max"]), step=None, layout=W.Layout(width="96px")))
                r_lo, r_hi = mk(lo), mk(hi)
                r_n = W.BoundedIntText(value={"coils": 41, "link": 60, "criteria": 3}.get(job, 5), min=1, max=4096,
                                       layout=W.Layout(width="70px"))
                r_log = W.Checkbox(value=bool(pd.get("log")), description="log", indent=False, layout=W.Layout(width="60px"))
                r_n.observe(lambda ch: on_change(), names="value")
                box = W.HBox([W.HTML("<span style='color:%s'>&nbsp;&nbsp;from</span>" % C["t3"]), r_lo,
                              W.HTML("<span style='color:%s'>to</span>" % C["t3"]), r_hi,
                              W.HTML("<span style='color:%s'>n</span>" % C["t3"]), r_n, r_log],
                             layout=W.Layout(display="none", align_items="center"))
                self.rng[pd["key"]] = (box, r_lo, r_hi, r_n, r_log)
                rows.append(box)
        self.box = W.VBox(rows)

    def _toggle(self, key, axis, on, on_change):
        boxes, other = (self.sx, self.sy) if axis == "x" else (self.sy, self.sx)
        if on:
            for k, cb in boxes.items():
                if k != key and cb.value:
                    cb.value = False
            if key in other and other[key].value:
                other[key].value = False
            if axis == "y":
                self.rng[key][3].value = 21
        for k, (box, _, _, _, _) in self.rng.items():
            box.layout.display = "flex" if ((k in self.sx and self.sx[k].value) or (k in self.sy and self.sy[k].value)) else "none"
        on_change()

    def _sweep(self, boxes):
        for k, cb in boxes.items():
            if cb.value:
                _, lo, hi, n, lg = self.rng[k]
                return {"key": k, "min": float(lo.value), "max": float(hi.value), "n": int(n.value),
                        "scale": "log" if lg.value else "lin"}
        return None

    def request(self):
        return {"job": self.job, "params": {k: e.value for k, e in self.ed.items()},
                "sweep": self._sweep(self.sx), "sweep2": self._sweep(self.sy)}


class StellaratorDashboard(object):
    """Notebook version of stellarator_ui.py: ipywidgets controls, plotly figures
    (3D machine view, step trace, sweep result, detail), results table and log.
    The job runs in the LiveJobRunner background thread; a second thread pumps its
    events into the widgets, so Run / Stop stay responsive while it computes."""

    def __init__(self, machine, runner):
        self.m, self.r = machine, runner
        self.schema = U._ui_schema(machine)
        self.geo = machine.geometry()
        self.S = None
        self.frame = {}
        self.ev_pos, self.ev_job = 0, -1
        self.lock = threading.RLock()
        self.dirty = {"charts": False, "view": False}
        self.last_draw = {"charts": 0.0, "view": 0.0}
        self.alive = True
        self.log_lines = []
        self._build()
        self.pump = threading.Thread(target=self._pump, daemon=True)
        self.pump.start()

    # ------------------------------------------------------------ layout
    def _build(self):
        g = self.geo
        self.forms = {j: ParamForm(j, s, self._estimate) for j, s in self.schema.items()}
        self.job = W.Dropdown(options=[(s["label"], j) for j, s in self.schema.items()], value="ness",
                              layout=W.Layout(width="100%"))
        self.about = W.HTML()
        self.form_box = W.VBox()
        self.est = W.HTML()
        self.run_b = W.Button(description="Run", button_style="primary", layout=W.Layout(width="48%"))
        self.stop_b = W.Button(description="Stop", layout=W.Layout(width="48%"))
        self.csv_b = W.Button(description="Export CSV", layout=W.Layout(width="48%"))
        self.json_b = W.Button(description="Export JSON", layout=W.Layout(width="48%"))
        self.run_b.on_click(lambda b: self.run_job())
        self.stop_b.on_click(lambda b: self.r.stop())
        self.csv_b.on_click(lambda b: self.export("csv"))
        self.json_b.on_click(lambda b: self.export("json"))
        self.job.observe(lambda ch: self._job_changed(), names="value")
        left = W.VBox([self.job, self.about, self.form_box, self.est,
                       W.HBox([self.run_b, self.stop_b]), W.HBox([self.csv_b, self.json_b])],
                      layout=W.Layout(width="420px", min_width="400px", padding="6px"))

        self.status = W.HTML()
        self.bar_p = W.FloatProgress(value=0, min=0, max=1, layout=W.Layout(width="100%", height="10px"))
        self.bar_s = W.FloatProgress(value=0, min=0, max=1, layout=W.Layout(width="100%", height="10px"),
                                     style={"bar_color": C["s3"]})
        self.bar_p.style.bar_color = C["s1"]
        header = W.VBox([W.HTML("<b style='font-size:15px'>W7-X Kitaev Stellarator - Live</b> "
                                "<span style='color:%s'>stellarator_colab.ipynb - 1080 sites - 40 x 27-qubit manifolds</span>" % C["t3"]),
                         self.bar_p, self.bar_s, self.status])

        # 3D view controls and figure
        self.v_colour = W.Dropdown(options=[("Live job data", "live"), ("Coupling J (coil field)", "weight"),
                                            ("Flux shell", "shell"), ("27-qubit block type", "block"),
                                            ("Half-period module", "module"), ("Kitaev bond colour x/y/z", "colour")],
                                   value="live", description="Colour", layout=W.Layout(width="290px"))
        self.v_shells = [W.Checkbox(value=True, description="s%d" % k, indent=False, layout=W.Layout(width="46px")) for k in range(5)]
        self.v_phi0 = W.IntSlider(value=0, min=0, max=360, description="Cut start", continuous_update=False,
                                  layout=W.Layout(width="260px"))
        self.v_span = W.IntSlider(value=360, min=20, max=360, description="Span", continuous_update=False,
                                  layout=W.Layout(width="260px"))
        self.v_layers = {k: W.Checkbox(value=True, description=lab, indent=False, layout=W.Layout(width=w))
                         for k, lab, w in (("bonds", "bonds", "70px"), ("sites", "sites", "62px"), ("coils", "coils", "62px"),
                                           ("div", "divertor", "84px"), ("over", "islands / seams / block", "190px"))}
        self.v_focus = W.Button(description="Focus job region", layout=W.Layout(width="150px"))
        self.v_reset = W.Button(description="Reset view", layout=W.Layout(width="110px"))
        for w_ in [self.v_colour, self.v_phi0, self.v_span] + self.v_shells + list(self.v_layers.values()):
            w_.observe(lambda ch: self._view_dirty(), names="value")
        self.v_focus.on_click(lambda b: self.focus())
        self.v_reset.on_click(lambda b: self.reset_camera())
        self.v_info = W.HTML()
        self.view = go.FigureWidget()
        self._init_view()
        view_card = W.VBox([
            W.HBox([self.v_colour, W.HTML("&nbsp;Shells&nbsp;")] + self.v_shells + [self.v_phi0, self.v_span],
                   layout=W.Layout(flex_flow="row wrap", align_items="center")),
            W.HBox(list(self.v_layers.values()) + [self.v_focus, self.v_reset],
                   layout=W.Layout(flex_flow="row wrap", align_items="center")),
            self.view, self.v_info])

        # charts
        self.tr_metric = W.Dropdown(layout=W.Layout(width="230px"))
        self.sw_metric = W.Dropdown(layout=W.Layout(width="230px"))
        self.tr_metric.observe(lambda ch: self._charts_dirty(), names="value")
        self.sw_metric.observe(lambda ch: self._charts_dirty(), names="value")
        self.tr_title, self.sw_title, self.dt_title = W.HTML(), W.HTML(), W.HTML()
        self.tr_fig = go.FigureWidget(layout=dict(height=320, **DARK))
        self.sw_fig = go.FigureWidget(layout=dict(height=320, **DARK))
        self.dt_fig = go.FigureWidget(layout=dict(height=300, **DARK))
        self.tr_leg, self.sw_leg = W.HTML(), W.HTML()
        # empty-state messages live in HTML, not plotly annotations (FigureWidget keeps stale annotations)
        nl = lambda: W.HTML(layout=W.Layout(height="0px", overflow="visible"))
        self.notes = {id(self.tr_fig): nl(), id(self.sw_fig): nl(), id(self.dt_fig): nl()}
        self.dt_html = W.HTML()
        self.dt_box = W.Box([self.dt_fig])   # hidden when the detail is a plain table
        self.table = W.HTML(layout=W.Layout(max_height="330px", overflow="auto"))
        self.table_hint = W.HTML()
        self.log = W.HTML(layout=W.Layout(max_height="170px", overflow="auto"))
        card = lambda kids: W.VBox(kids, layout=W.Layout(width="50%", padding="4px", border="1px solid %s" % C["border"]))
        charts = W.VBox([
            W.HBox([card([W.HBox([self.tr_title, self.tr_metric], layout=W.Layout(justify_content="space-between")), self.notes[id(self.tr_fig)], self.tr_fig, self.tr_leg]),
                    card([W.HBox([self.sw_title, self.sw_metric], layout=W.Layout(justify_content="space-between")), self.notes[id(self.sw_fig)], self.sw_fig, self.sw_leg])]),
            W.HBox([card([self.dt_title, self.notes[id(self.dt_fig)], self.dt_box, self.dt_html]),
                    card([W.HBox([W.HTML("<b>Results table</b>"), self.table_hint], layout=W.Layout(justify_content="space-between")), self.table])]),
            W.VBox([W.HTML("<b>Log</b>"), self.log], layout=W.Layout(padding="4px", border="1px solid %s" % C["border"])),
        ])
        right = W.VBox([view_card, charts], layout=W.Layout(flex="1 1 auto", min_width="0"))
        style = W.HTML("""<style>
          .stl-root, .stl-root .widget-label, .stl-root .widget-html-content { color: %(t1)s; }
          .stl-root { background: %(page)s; padding: 8px; border-radius: 8px; }
          .stl-root table { border-collapse: collapse; font-size: 12px; font-variant-numeric: tabular-nums; }
          .stl-root th, .stl-root td { padding: 3px 8px; border-bottom: 1px solid %(border)s; text-align: right; white-space: nowrap; }
          .stl-root th { color: %(t2)s; font-weight: 500; }
          .stl-root label, .stl-root .widget-checkbox span, .stl-root .widget-label-basic,
          .stl-root .widget-label-basic span, .stl-root .widget-readout { color: %(t2)s !important; }
          .stl-root td:first-child, .stl-root th:first-child { text-align: left; }
        </style>""" % C)
        self.root = W.VBox([style, header, W.HBox([left, right], layout=W.Layout(width="100%", overflow_x="auto"))])
        self.root.add_class("stl-root")
        self._job_changed()
        self.render_charts()

    def show(self):
        if "google.colab" in sys.modules:
            from google.colab import output
            output.enable_custom_widget_manager()
        from IPython.display import display
        display(self.root)

    # ------------------------------------------------------------ helpers
    def _job_changed(self):
        j = self.job.value
        spec = self.schema[j]
        self.about.value = "<b>%s</b><br><span style='color:%s;font-size:12px'>%s</span>" % (spec["title"], C["t2"], spec["about"])
        self.form_box.children = [self.forms[j].box]
        if not self.S or not self.S["running"]:
            self._set_metrics(j)
            self._charts_dirty()
        self._estimate()

    def _set_metrics(self, job):
        spec = self.schema[job]
        tm = [(LAB(k), k) for k in spec["step_metrics"]]
        self.tr_metric.options = tm or [("-", None)]
        self.tr_metric.value = spec.get("default_step_metric") if tm else None
        self.tr_metric.layout.display = "flex" if tm else "none"
        self.sw_metric.options = [(LAB(k), k) for k in spec["summary_metrics"]]
        self.sw_metric.value = spec["default_summary_metric"]

    def _estimate(self):
        j = self.job.value
        r = self.forms[j].request()
        n = 1
        for s_ in (r["sweep"], r["sweep2"]):
            if s_:
                n *= max(1, s_["n"])
        p = r["params"]
        per = {"ness": 0.35 * float(p.get("steps", 135)), "coils": 0.003, "criteria": 62.0,
               "manifold": (5.0 if str(p.get("qubits", "")).startswith("27") else 0.07) * float(p.get("steps", 20)),
               "link": 0.001}[j]
        self.est.value = "<span style='color:%s'>%d point%s - estimated %s</span>" % (C["t3"], n, "" if n == 1 else "s", DUR(n * per))

    def _charts_dirty(self):
        self.dirty["charts"] = True

    def _view_dirty(self):
        self.dirty["view"] = True

    def log_line(self, msg, err=False):
        ts = time.strftime("%H:%M:%S")
        safe = str(msg).replace("&", "&amp;").replace("<", "&lt;").replace("\n", "<br>")
        self.log_lines.append("<div style='color:%s;font-family:monospace;font-size:11px'>%s&nbsp;&nbsp;%s</div>"
                              % (C["crit"] if err else C["t2"], ts, safe))
        self.log_lines = self.log_lines[-300:]
        self.log.value = "".join(self.log_lines[::-1])

    @staticmethod
    def coord_text(c):
        return "(" + ", ".join("%s=%s" % (k, FMT(v)) for k, v in c.items()) + ")" if c else ""

    # ------------------------------------------------------------ actions
    def run_job(self, req=None):
        req = req or self.forms[self.job.value].request()
        try:
            self.r.start(req)
        except (ValueError, RuntimeError) as exc:
            self.log_line("Run refused: %s" % exc, err=True)

    def export(self, kind):
        S = self.S
        if not S:
            return
        if kind == "json":
            path = "stellarator_ui_results.json"
            with open(path, "w") as fh:
                json.dump(self.r.last_results or {}, fh, indent=1, default=U._ui_json_default)
        else:
            path = "stellarator_ui_%s.csv" % S["job"]
            keys = self.schema[S["job"]]["summary_metrics"]
            axk = [a["key"] for a in S["axes"]]
            with open(path, "w", newline="") as fh:
                wr = csv.writer(fh)
                wr.writerow(["point"] + axk + keys + ["seconds"])
                for i in S["order"]:
                    P = S["points"][i]
                    if P["summary"] is not None:
                        wr.writerow([i + 1] + [P["coords"][k] for k in axk] + [P["summary"].get(k) for k in keys] + [P.get("seconds")])
        self.log_line("Saved %s" % path)
        try:
            from google.colab import files
            files.download(path)
        except Exception:
            pass

    # ------------------------------------------------------------ event pump
    def _pump(self):
        while self.alive:
            try:
                self.poll()
            except Exception as exc:  # keep the pump alive, report once
                self.log_line("UI update error: %s" % exc, err=True)
                time.sleep(1.0)
            time.sleep(0.15)

    def poll(self):
        with self.r.cond:
            if self.r.job_id != self.ev_job:
                self.ev_job, self.ev_pos = self.r.job_id, 0
            batch = self.r.events[self.ev_pos:]
            self.ev_pos += len(batch)
        with self.lock:
            for ev in batch:
                self.on_event(ev)
            now = time.time()
            running = bool(self.S and self.S["running"])
            if self.S:
                self.render_header()
            if self.dirty["view"] and now - self.last_draw["view"] > (0.6 if running else 0.05):
                self.render_view()
            if self.dirty["charts"] and now - self.last_draw["charts"] > (0.8 if running else 0.05):
                self.render_charts()
            self.run_b.disabled = self.r.status()["running"]

    def on_event(self, ev):
        t = ev["type"]
        if t == "job_start":
            self.S = {"job": ev["job"], "job_id": ev["job_id"], "axes": ev["axes"], "n_points": ev["n_points"],
                      "points": {}, "order": [], "cur": -1, "step": 0, "n_steps": 0, "status": "running",
                      "running": True, "t0": ev["t"], "t_last": ev["t"], "last_step": None, "phases": [], "done": 0}
            self.frame = {}
            if self.job.value != ev["job"]:
                self.job.value = ev["job"]
            self._set_metrics(ev["job"])
            axs = (" - sweep " + " x ".join("%s [%d]" % (a["key"], len(a["values"])) for a in ev["axes"])) if ev["axes"] else ""
            self.log_line("Job %d started: %s - %d point(s)%s" % (ev["job_id"], self.schema[ev["job"]]["title"], ev["n_points"], axs))
            self.dirty.update(charts=True, view=True)
            return
        S = self.S
        if not S or ev["job_id"] != S["job_id"]:
            return
        S["t_last"] = ev["t"]
        if t == "point_start":
            S["cur"] = ev["i"]
            S["points"][ev["i"]] = {"coords": ev["coords"], "steps": [], "summary": None}
            S["order"].append(ev["i"])
            S["step"], S["phases"] = 0, []
        elif t == "step":
            P = S["points"].get(ev["i"])
            if P is None:
                return
            S["step"], S["n_steps"], S["last_step"] = ev["step"], ev["n_steps"], ev["data"]
            if S["job"] == "criteria":
                S["phases"].append((ev["data"]["phase"], ev["t"]))
            else:
                P["steps"].append((ev["step"], {k: ev["data"].get(k) for k in self.schema[S["job"]]["step_metrics"]}))
        elif t == "frame":
            if ev.get("frame"):
                self.frame.update(ev["frame"])
                if "bond" in ev["frame"]:
                    self.frame["_step"] = S["step"]
                self.dirty["view"] = True
            return
        elif t == "point_done":
            P = S["points"].get(ev["i"])
            if P is None:
                return
            P["summary"], P["seconds"] = ev["summary"], ev["seconds"]
            S["done"] += 1
            if S["job"] in ("ness", "manifold", "criteria"):
                keys = {"ness": ["tau_E", "ex_in_ratio", "per_p2m"], "manifold": ["max_rms", "final_rms", "W0"],
                        "criteria": ["beta_crit_pct", "shafranov_shift_5pct", "fast_ion_survival"]}[S["job"]]
                self.log_line("Point %d/%d %s done in %.1f s: %s" % (ev["i"] + 1, S["n_points"], self.coord_text(ev["coords"]),
                              ev["seconds"], ", ".join("%s %s" % (LAB(k), FMT(ev["summary"].get(k))) for k in keys)))
        elif t == "log":
            self.log_line(ev["msg"])
        elif t in ("job_done", "job_cancelled", "job_error"):
            S["running"] = False
            S["status"] = {"job_done": "done", "job_cancelled": "cancelled", "job_error": "error"}[t]
            if t == "job_error":
                self.log_line("ERROR %s\n%s" % (ev["error"], ev.get("trace", "")), err=True)
            else:
                self.log_line("Job %d %s after %s (%d point(s))" % (S["job_id"], S["status"], DUR(ev["seconds"]), ev["n_points"]))
            self.dirty["view"] = True
        self.dirty["charts"] = True

    def wait(self, timeout=3600.0):
        """Block until the running job ends and the widgets show its final state."""
        t0 = time.time()
        time.sleep(0.3)
        while self.r.status()["running"] and time.time() - t0 < timeout:
            time.sleep(0.2)
        time.sleep(0.5)
        with self.lock:
            self.poll()
            self.render_charts()
            self.render_view()

    # ------------------------------------------------------------ header
    def render_header(self):
        S = self.S
        col = {"running": C["s1"], "done": C["good"], "cancelled": C["warn"], "error": C["crit"]}[S["status"]]
        frac = (S["done"] + (S["step"] / S["n_steps"] if S["running"] and S["n_steps"] else 0)) / max(1, S["n_points"])
        self.bar_p.value = frac if S["running"] else S["done"] / max(1, S["n_points"])
        self.bar_s.value = S["step"] / S["n_steps"] if S["n_steps"] else 0
        el = (time.time() if S["running"] else S["t_last"]) - S["t0"]
        eta = el * (1 - frac) / frac if S["running"] and frac > 0.002 else 0
        cur = S["points"].get(S["cur"])
        txt = ("<span style='border:1px solid %s;color:%s;border-radius:9px;padding:0 8px;font-weight:600'>%s</span>&nbsp; "
               % (col, col, S["status"]))
        txt += "Job %d - point %d / %d %s" % (S["job_id"], min(S["done"] + (1 if S["running"] else 0), S["n_points"]),
                                            S["n_points"], self.coord_text(cur["coords"]) if cur else "")
        if S["n_steps"]:
            txt += " &nbsp; %s %d / %d" % ("phase" if S["job"] == "criteria" else "step", S["step"], S["n_steps"])
        txt += " &nbsp; elapsed %s" % DUR(el) + (" &nbsp; ETA %s" % DUR(eta) if eta else "")
        self.status.value = "<span style='color:%s'>%s</span>" % (C["t2"], txt)

    # ------------------------------------------------------------ 3D view
    def _camera(self, center=(0, 0, 0), eye=(1.15, -1.15, 0.55)):
        return dict(center=dict(x=center[0], y=center[1], z=center[2]), eye=dict(x=eye[0], y=eye[1], z=eye[2]))

    VIEW_TRACES = ["coils", "base", "bonds", "isl", "seams", "sites", "hl", "div"]

    def _blank_traces(self):
        out = []
        for nm in self.VIEW_TRACES:
            if nm in ("sites", "hl", "div"):
                out.append(go.Scatter3d(x=[], y=[], z=[], mode="markers", name=nm, hoverinfo="text", text=[]))
            else:
                out.append(go.Scatter3d(x=[], y=[], z=[], mode="lines", name=nm, hoverinfo="skip", connectgaps=False))
        return out

    def _init_view(self):
        g = self.geo
        ax = dict(visible=False, showbackground=False)
        self.view.layout = go.Layout(height=600, paper_bgcolor="#161615", margin=dict(l=0, r=0, t=0, b=0),
                                     showlegend=False, uirevision="keep",
                                     scene=dict(xaxis=ax, yaxis=ax, zaxis=ax, aspectmode="data",
                                                bgcolor="#161615", camera=self._camera()),
                                     font=DARK["font"])
        self.view.add_traces(self._blank_traces())
        self.tr_idx = {nm: k for k, nm in enumerate(self.VIEW_TRACES)}
        P = g["P"]
        cx, cy, cz, cf = [], [], [], []
        for c in g["coils"]:
            xyz = c["xyz"]
            cx += list(xyz[:, 0]) + [np.nan]
            cy += list(xyz[:, 1]) + [np.nan]
            cz += list(xyz[:, 2]) + [np.nan]
        self.coil_xyz = (np.array(cx), np.array(cy), np.array(cz))
        self.coil_len = [len(c["xyz"]) + 1 for c in g["coils"]]
        self.site_text = ["site %d<br>shell s%d, module %d, block %d" % (i, g["shell"][i], g["module"][i], g["block"][i])
                          + ("<br>core axis site" if i in g["core"] else "") + ("<br>divertor target" if i in g["div"] else "")
                          for i in range(g["n"])]
        self.render_view()

    def reset_camera(self):
        self.view.layout.scene.camera = self._camera()

    def focus(self):
        g, f3 = self.geo, self.frame
        ids = list(f3.get("highlight") or [])
        if not ids and f3.get("isl_bonds"):
            b = g["bonds"][f3["isl_bonds"][0]]
            ids = [b[0], b[1]]
        if ids:
            t = g["P"][ids].mean(axis=0)
        elif f3.get("units"):
            t = g["units"][int(np.argmax(f3["units"]))]
        else:
            return
        span = np.ptp(g["P"], axis=0)
        ctr = (t - g["P"].mean(axis=0)) / span.max()
        r = math.hypot(t[0], t[1])
        eye = (ctr[0] + 0.25 * t[0] / max(r, 1e-9), ctr[1] + 0.25 * t[1] / max(r, 1e-9), ctr[2] + 0.12)
        self.v_span.value = 360
        self.view.layout.scene.camera = self._camera(tuple(ctr), eye)

    def _live_values(self):
        f, mode = self.frame, self.v_colour.value
        if mode == "weight":
            w = f.get("weights") or (f.get("bond") if f.get("bond_mode") == "weight" else None)
            return (w, None, "weight", "Coupling J") if w else None
        if mode != "live" or not f.get("bond"):
            return None
        lab = {"energy": "Excess bond energy", "weight": "Coupling J", "error": "PyQrack bond error"}.get(f.get("bond_mode"), "value")
        return (f["bond"], f.get("bond_ids"), f.get("bond_mode"), lab)

    def render_view(self):
        with self.lock:
            self.last_draw["view"] = time.time()
            self.dirty["view"] = False
            g, f3, mode = self.geo, self.frame, self.v_colour.value
            P, bonds = g["P"], g["bonds"]
            phi0, span = math.radians(self.v_phi0.value), math.radians(self.v_span.value)
            shells_on = np.array([cb.value for cb in self.v_shells])
            win = (np.ones(g["n"], bool) if span >= 2 * math.pi - 1e-6
                   else np.mod(g["phi"] - phi0 + 4 * math.pi, 2 * math.pi) <= span)
            vis = shells_on[g["shell"]] & win
            bvis = vis[g["bi"]] & vis[g["bj"]]
            # Fill fresh traces and swap them in whole. Restyling WebGL line traces in place
            # (changing point counts and colour arrays) left stale GPU buffers in the widget
            # frontend: wrong colours and ghost geometry.
            T = go.Figure(data=self._blank_traces()).data
            ti = self.tr_idx
            info = []
            with contextlib.nullcontext():
                # coils
                if self.v_layers["coils"].value:
                    cf = f3.get("coils")
                    vals, cmax = [], 1.0
                    coil_win = (np.ones(len(g["coils"]), bool) if span >= 2 * math.pi - 1e-6
                                else np.mod(g["coil_phi"] - phi0 + 4 * math.pi, 2 * math.pi) <= span)
                    devs = []
                    for c in g["coils"]:
                        if not cf:
                            devs.append(0.0)
                        elif c["fam"] == "non_planar":
                            devs.append(cf["np"][c["type"]] - cf["np0"][c["type"]])
                        elif c["fam"] == "planar":
                            devs.append(cf["pc"][c["type"]] - cf["pc0"][c["type"]])
                        elif c["fam"] == "trim":
                            devs.append(cf["trim"][c["k"]])
                        else:
                            devs.append(cf["ctrl"][c["k"]])
                    fam_max = {}
                    for c, d in zip(g["coils"], devs):
                        fam_max[c["fam"]] = max(fam_max.get(c["fam"], 0.0), abs(d))
                    for c, d, n_, ok in zip(g["coils"], devs, self.coil_len, coil_win):
                        floor_ = 0.05 if c["fam"] in ("non_planar", "planar") else 1e-3
                        vals += [d / max(fam_max[c["fam"]], floor_) if ok else np.nan] * n_
                    mask = np.repeat(coil_win, self.coil_len)
                    x, y, z = (np.where(mask, a, np.nan) for a in self.coil_xyz)
                    vals = [0.0 if not np.isfinite(v) else v for v in vals]
                    T[ti["coils"]].update(x=x, y=y, z=z, opacity=1.0,
                                          line=dict(color=vals if cf else _dim("#5a5955", 0.3), colorscale=CS_DIV3D_DIM, cmin=-1, cmax=1, width=2))
                    info.append("Coils: current vs preset nominal (blue below, red above)" if cf else "Coils: 50 non-planar, 20 planar, 5 trim, 10 control")
                else:
                    T[ti["coils"]].update(x=[], y=[], z=[])
                # bonds: dim base lattice plus coloured live / category bonds
                lv = self._live_values() if mode in ("live", "weight") else None
                show_b = self.v_layers["bonds"].value
                ids_all = np.nonzero(bvis)[0]
                if show_b:
                    x, y, z = _segments(P, bonds, ids_all)
                    T[ti["base"]].update(x=x, y=y, z=z, line=dict(color=_dim("#3a3a37", 0.6), width=1.5), opacity=1.0)
                else:
                    T[ti["base"]].update(x=[], y=[], z=[])
                cat = None
                if mode == "shell":
                    cat = (np.maximum(g["shell"][g["bi"]], g["shell"][g["bj"]]), U.UI_CAT8[:5], "Flux shell s0 to s4")
                elif mode == "block":
                    bt = np.mod(np.maximum(g["block"][g["bi"]], 0), 4).astype(float)
                    bt[list(g["seam_set"])] = 4
                    cat = (bt, U.UI_CAT8[:4] + ["#f0efec"], "27-qubit block type 0 to 3, seams white")
                elif mode == "module":
                    cat = (g["module"][g["bi"]] % 10, U.UI_MODRAMP, "Half-period module 0 to 9")
                elif mode == "colour":
                    cat = (g["bcolour"] % 3, U.UI_CAT8[:3], "Kitaev bond colour x (blue), y (orange), z (aqua)")
                if show_b and cat is not None:
                    cv, ramp, lab = cat
                    ids = ids_all
                    x, y, z = _segments(P, bonds, ids)
                    cols = np.repeat(cv[ids].astype(float), 3).tolist()
                    n_c = len(ramp)
                    cs = []
                    for k, c in enumerate(ramp):
                        cs += [[k / n_c, c], [(k + 1) / n_c - 1e-9, c]]
                    cs[-1][0] = 1.0
                    T[ti["bonds"]].update(x=x, y=y, z=z, opacity=1.0,
                                          line=dict(color=cols, colorscale=cs, cmin=-0.5, cmax=n_c - 0.5, width=3))
                    info.insert(0, lab)
                elif show_b and lv:
                    vals = np.full(len(bonds), np.nan)
                    v = np.asarray(lv[0], dtype=float)
                    if lv[1]:
                        vals[np.asarray(lv[1], dtype=int)] = v
                    else:
                        vals[:len(v)] = v
                    ids = ids_all[np.isfinite(vals[ids_all])]
                    fin = vals[ids]
                    if len(ids):
                        if lv[2] == "weight":
                            m = max(1e-6, float(np.max(np.abs(fin - 1.0))))
                            cs, cmin, cmax, cval, rng = CS_DIV3D, 1 - m, 1 + m, fin, (1 - m, 1 + m)
                        elif lv[2] == "error":
                            lo = math.log10(max(1e-9, float(np.quantile(fin, 0.02))))
                            hi = max(math.log10(max(1e-9, float(fin.max()))), lo + 0.5)
                            cs, cmin, cmax, cval, rng = CS_SEQ, lo, hi, np.log10(np.maximum(fin, 1e-12)), (10 ** lo, 10 ** hi)
                        else:
                            lo = min(0.0, float(np.quantile(fin, 0.02)))
                            hi = max(float(np.quantile(fin, 0.995)), lo + 1e-6)
                            cs, cmin, cmax, cval, rng = CS_ORANGE, lo, hi, fin, (lo, hi)
                        x, y, z = _segments(P, bonds, ids)
                        T[ti["bonds"]].update(x=x, y=y, z=z, opacity=1.0,
                                              line=dict(color=np.repeat(cval, 3).tolist(), colorscale=cs, cmin=cmin, cmax=cmax, width=5))
                        ramp = {"weight": "blue-gray-red", "error": "dark to light blue, log", "energy": "dark to light orange"}.get(lv[2], "")
                        info.insert(0, "%s %s to %s (%s)" % (lv[3], FMT(rng[0]), FMT(rng[1]), ramp))
                    else:
                        T[ti["bonds"]].update(x=[], y=[], z=[])
                else:
                    T[ti["bonds"]].update(x=[], y=[], z=[])
                    if mode == "live":
                        info.insert(0, "Run a job to colour the machine with live data")
                    elif mode == "weight":
                        info.insert(0, "Run a NESS, coil or 7-criteria job to load coupling weights")
                # overlays
                over = self.v_layers["over"].value
                isl = [k for k in (f3.get("isl_bonds") or []) if bvis[k]] if over else []
                x, y, z = _segments(P, bonds, isl)
                T[ti["isl"]].update(x=x, y=y, z=z, line=dict(color=C["s5"], width=9))
                if isl:
                    info.append("%d edge-island flux defects (flipped u, magenta)" % len(isl))
                sm = [k for k in (f3.get("seams") or []) if bvis[k]] if over else []
                x, y, z = _segments(P, bonds, sm)
                T[ti["seams"]].update(x=x, y=y, z=z, line=dict(color=_dim("#f0efec", 0.8), width=3), opacity=1.0)
                if sm:
                    info.append("%d seam bonds (1 ebit per cut per step, white)" % len(sm))
                hl = [i for i in (f3.get("highlight") or []) if vis[i]] if over else []
                T[ti["hl"]].update(x=P[hl, 0] if hl else [], y=P[hl, 1] if hl else [], z=P[hl, 2] if hl else [],
                                   marker=dict(size=7, symbol="circle-open", color=C["s2"], line=dict(color=C["s2"], width=3)),
                                   text=[self.site_text[i] + "<br>PyQrack qubit" for i in hl])
                if hl:
                    info.append("%d PyQrack qubits (orange rings)" % len(hl))
                # sites
                if self.v_layers["sites"].value:
                    sid = np.nonzero(vis)[0]
                    cols, sizes = [], []
                    for i in sid:
                        if i in g["core"]:
                            cols.append(C["s3"]); sizes.append(3.5)
                        elif i in g["div"]:
                            cols.append(C["s6"]); sizes.append(3.2)
                        elif mode == "shell":
                            cols.append(U.UI_CAT8[g["shell"][i]]); sizes.append(2.2)
                        elif mode == "block":
                            cols.append(U.UI_CAT8[max(0, g["block"][i]) % 4]); sizes.append(2.2)
                        elif mode == "module":
                            cols.append(U.UI_MODRAMP[g["module"][i] % 10]); sizes.append(2.2)
                        else:
                            cols.append("#8f8e85"); sizes.append(1.8)
                    T[ti["sites"]].update(x=P[sid, 0], y=P[sid, 1], z=P[sid, 2],
                                          marker=dict(size=sizes, color=[_dim(c, 0.85) for c in cols], opacity=1.0, line=dict(width=0)),
                                          text=[self.site_text[i] for i in sid])
                else:
                    T[ti["sites"]].update(x=[], y=[], z=[], text=[])
                # divertor units
                if self.v_layers["div"].value:
                    u = f3.get("units")
                    U3 = g["units"]
                    uphi = np.mod(np.arctan2(U3[:, 1], U3[:, 0]), 2 * math.pi)
                    uw = (np.ones(len(U3), bool) if span >= 2 * math.pi - 1e-6
                          else np.mod(uphi - phi0 + 4 * math.pi, 2 * math.pi) <= span)
                    ks = np.nonzero(uw)[0]
                    sz = [10 if not u else 6 + 16 * max(0.0, u[k]) for k in ks]
                    T[ti["div"]].update(x=U3[ks, 0], y=U3[ks, 1], z=U3[ks, 2] + 0.0, mode="markers+text",
                                        marker=dict(size=sz, color=_dim(C["s3"], 0.45), line=dict(color=C["s3"], width=2)),
                                        text=["D%d%s" % (k + 1, "" if not u else " %.2f" % u[k]) for k in ks],
                                        textfont=dict(color=C["t2"], size=10), textposition="top center")
                    info.append("divertor units (size = exhaust / mean)")
                else:
                    T[ti["div"]].update(x=[], y=[], z=[], text=[])
            # add the new set first, then drop the old one, so the scene never goes empty (no 2D flash)
            n_old = len(self.view.data)
            self.view.add_traces(list(T))
            self.view.data = self.view.data[n_old:]
            hud = f3.get("label") or ""
            if self.S and self.S["n_steps"] and f3.get("_step"):
                hud += " - step %d / %d" % (f3["_step"], self.S["n_steps"])
            self.v_info.value = "<span style='color:%s;font-size:12px'>%s%s</span>" % (
                C["t3"], ("<b style='color:%s'>%s</b><br>" % (C["t2"], hud)) if hud else "", " &nbsp;|&nbsp; ".join(info))

    # ------------------------------------------------------------ charts
    def render_charts(self):
        with self.lock:
            self.last_draw["charts"] = time.time()
            self.dirty["charts"] = False
            for n in self.notes.values():
                if n.value:
                    n.value = ""
            self.render_trace()
            self.render_sweep()
            self.render_detail()
            self.render_table()

    def _axis_info(self):
        S = self.S
        if not S or not S["axes"]:
            return None
        a = S["axes"][0]
        v = a["values"]
        return {"key": a["key"], "lo": min(v), "hi": max(v), "log": a.get("scale") == "log" and min(v) > 0}

    def _empty(self, fig, msg, height=320):
        fig.data = []
        fig.layout = go.Layout(height=height, **DARK)
        fig.layout.xaxis.visible = False
        fig.layout.yaxis.visible = False
        self.notes[id(fig)].value = ("<div style='position:relative;top:%dpx;text-align:center;color:%s;"
                                     "font-size:12px;z-index:5;pointer-events:none'>%s</div>") % (height // 2 - 10, C["t3"], msg)

    def render_trace(self):
        S, fig = self.S, self.tr_fig
        job = S["job"] if S else self.job.value
        spec = self.schema[job]
        with fig.batch_update():
            if job == "criteria":
                self.tr_title.value = "<b>Evaluation phases</b>"
                self.tr_leg.value = ""
                if not S:
                    return self._empty(fig, "Run the 7-criteria evaluation to follow its phases here")
                seen = [p[0] for p in S["phases"] if p[0] != "done"]
                cur_done = S["points"].get(S["cur"], {}).get("summary") is not None if S["cur"] >= 0 else False
                durs, cols = [], []
                for lb in U.UI_PHASES:
                    if lb in seen:
                        idx = seen.index(lb)
                        t_end = S["phases"][idx + 1][1] if idx + 1 < len(S["phases"]) else (S["t_last"] if cur_done else time.time())
                        durs.append(t_end - S["phases"][idx][1])
                        cols.append(C["s2"] if idx == len(seen) - 1 and S["running"] and not cur_done else C["s1"])
                    else:
                        durs.append(0.0)
                        cols.append(C["mid"])
                fig.data = []
                fig.add_trace(go.Bar(x=durs, y=U.UI_PHASES, orientation="h", marker_color=cols,
                                     text=["%.1f s" % d if d else "" for d in durs], textposition="outside",
                                     textfont=dict(color=C["t2"]), hovertemplate="%{y}: %{x:.1f} s<extra></extra>"))
                fig.layout = go.Layout(height=320, **DARK)
                fig.update_xaxes(title_text="seconds", **AX)
                fig.update_yaxes(autorange="reversed", **AX)
                fig.layout.margin.l = 170
                return
            if not spec["step_metrics"]:
                title, keys, ylabel = U.UI_PROFILE[job]
                self.tr_title.value = "<b>%s</b>" % title
                self.tr_leg.value = ""
                if not S or not S["axes"]:
                    return self._empty(fig, "Sweep a parameter to see the profile" if S else "Set parameters and press Run")
                ai = self._axis_info()
                pts = [S["points"][i] for i in S["order"] if S["points"][i]["summary"] is not None]
                note = ""
                if len(S["axes"]) == 2 and pts:
                    ky = S["axes"][1]["key"]
                    yv = pts[-1]["coords"][ky]
                    pts = [P for P in pts if P["coords"][ky] == yv]
                    note = " at %s = %s" % (LAB(ky), FMT(yv))
                pts.sort(key=lambda P: P["coords"][ai["key"]])
                fig.data = []
                xs = [P["coords"][ai["key"]] for P in pts]
                for n_, k in enumerate(keys):
                    fig.add_trace(go.Scatter(x=xs, y=[P["summary"].get(k) for P in pts], mode="lines", name=LAB(k),
                                             line=dict(color=U.UI_CAT8[n_], width=2)))
                fig.layout = go.Layout(height=320, showlegend=True, legend=dict(font=dict(size=10, color=C["t2"]), bgcolor="rgba(0,0,0,0)"), **DARK)
                fig.update_xaxes(title_text=LAB(ai["key"]) + note, type="log" if ai["log"] else "linear", **AX)
                fig.update_yaxes(title_text=ylabel, **AX)
                return
            self.tr_title.value = "<b>Live step trace</b>"
            m = self.tr_metric.value or spec["default_step_metric"]
            if not S:
                self.tr_leg.value = ""
                return self._empty(fig, "Set parameters and press Run")
            ai = self._axis_info()
            traces = []
            for i in S["order"]:
                P = S["points"][i]
                if not P["steps"]:
                    continue
                running = i == S["cur"] and S["running"]
                if running:
                    col, w = C["s2"], 3
                elif ai:
                    v = P["coords"][ai["key"]]
                    t = ((math.log10(v) - math.log10(ai["lo"])) / max(1e-12, math.log10(ai["hi"]) - math.log10(ai["lo"]))
                         if ai["log"] else (v - ai["lo"]) / max(1e-12, ai["hi"] - ai["lo"]))
                    col, w = _rgb(t, CS_TRACE), 2
                else:
                    col, w = C["s1"], 2
                nm = ("%s=%s" % (ai["key"], FMT(P["coords"][ai["key"]])) if ai else "point %d" % (i + 1)) + (" (running)" if running else "")
                traces.append((running, go.Scatter(x=[r[0] for r in P["steps"]], y=[r[1].get(m) for r in P["steps"]],
                                                   mode="lines", line=dict(color=col, width=w), name=nm)))
            traces.sort(key=lambda t: t[0])
            fig.data = []
            for _, tr in traces:
                fig.add_trace(tr)
            fig.layout = go.Layout(height=320, showlegend=False, hovermode="x unified", **DARK)
            fig.update_xaxes(title_text="step", **AX)
            fig.update_yaxes(title_text=LAB(m), type="log" if m in U.UI_LOGY else "linear", **AX)
            leg = ("%s %s to %s (dark to light)" % (LAB(ai["key"]), FMT(ai["lo"]), FMT(ai["hi"])) if ai else LAB(m))
            if S["running"]:
                leg += " &nbsp;|&nbsp; orange: running point"
            self.tr_leg.value = "<span style='color:%s;font-size:11px'>%s</span>" % (C["t3"], leg)

    def render_sweep(self):
        S, fig = self.S, self.sw_fig
        job = S["job"] if S else self.job.value
        m = self.sw_metric.value or self.schema[job]["default_summary_metric"]
        with fig.batch_update():
            if not S:
                self.sw_title.value = "<b>Sweep result</b>"
                self.sw_leg.value = ""
                return self._empty(fig, "Results appear here point by point")
            done = [S["points"][i] for i in S["order"] if S["points"][i]["summary"] is not None]
            if len(S["axes"]) == 2:
                self.sw_title.value = "<b>Sweep result (heatmap)</b>"
                xs, ys = S["axes"][0]["values"], S["axes"][1]["values"]
                grid = np.full((len(ys), len(xs)), np.nan)
                xi = {v: k for k, v in enumerate(xs)}
                yi = {v: k for k, v in enumerate(ys)}
                for P in done:
                    a, b = xi.get(P["coords"][S["axes"][0]["key"]]), yi.get(P["coords"][S["axes"][1]["key"]])
                    if a is not None and b is not None and P["summary"].get(m) is not None:
                        grid[b, a] = P["summary"][m]
                fin = grid[np.isfinite(grid)]
                diverge = len(fin) and fin.min() < 0 < fin.max()
                mm = float(np.abs(fin).max()) if len(fin) else 1.0
                fig.data = []
                fig.add_trace(go.Heatmap(x=xs, y=ys, z=grid, colorscale=CS_DIV if diverge else CS_SEQ,
                                         zmin=-mm if diverge else None, zmax=mm if diverge else None,
                                         colorbar=dict(tickfont=dict(color=C["t3"]), outlinewidth=0),
                                         hovertemplate=LAB(S["axes"][0]["key"]) + " %{x}<br>" + LAB(S["axes"][1]["key"]) + " %{y}<br>" + LAB(m) + " %{z:.4g}<extra></extra>"))
                fig.layout = go.Layout(height=320, **DARK)
                fig.update_xaxes(title_text=LAB(S["axes"][0]["key"]), **AX)
                fig.update_yaxes(title_text=LAB(S["axes"][1]["key"]), **AX)
                self.sw_leg.value = "<span style='color:%s;font-size:11px'>%s: %d / %d points</span>" % (C["t3"], LAB(m), len(done), S["n_points"])
            elif not S["axes"]:
                self.sw_title.value = "<b>Result</b>"
                self.sw_leg.value = ""
                fig.data = []
                if done:
                    fig.add_trace(go.Indicator(mode="number", value=done[0]["summary"].get(m) or 0,
                                               number=dict(font=dict(color=C["t1"], size=48), valueformat=".4g"),
                                               title=dict(text=LAB(m), font=dict(color=C["t2"], size=14))))
                    fig.layout = go.Layout(height=320, xaxis=dict(visible=False), yaxis=dict(visible=False), **DARK)
                else:
                    self._empty(fig, "Single-point run: value appears when it finishes")
            else:
                self.sw_title.value = "<b>Sweep result</b>"
                ai = self._axis_info()
                pts = sorted(done, key=lambda P: P["coords"][ai["key"]])
                fig.data = []
                fig.add_trace(go.Scatter(x=[P["coords"][ai["key"]] for P in pts], y=[P["summary"].get(m) for P in pts],
                                         mode="lines+markers" if len(pts) <= 80 else "lines",
                                         line=dict(color=C["s1"], width=2),
                                         marker=dict(size=8, color=C["s1"], line=dict(color=C["surf"], width=2)), name=LAB(m)))
                fig.layout = go.Layout(height=320, showlegend=False, **DARK)
                fig.update_xaxes(title_text=LAB(ai["key"]), type="log" if ai["log"] else "linear", **AX)
                fig.update_yaxes(title_text=LAB(m), type="log" if m in U.UI_LOGY else "linear", **AX)
                self.sw_leg.value = "<span style='color:%s;font-size:11px'>%s vs %s - %d / %d points</span>" % (
                    C["t3"], LAB(m), LAB(ai["key"]), len(done), S["n_points"])

    def render_detail(self):
        S, fig = self.S, self.dt_fig
        self.dt_box.layout.display = ""
        job = S["job"] if S else self.job.value
        with fig.batch_update():
            if not S:
                self.dt_title.value = "<b>Detail</b>"
                self.dt_html.value = ""
                fig.layout.height = 300
                return self._empty(fig, "Shell energies and divertor loads (NESS), bond agreement (PyQrack), last result for the others", 300)
            d = S["last_step"]
            if job == "ness":
                self.dt_title.value = "<b>Shell energy and divertor load (current step)</b>"
                self.dt_html.value = ""
                sub = make_subplots(rows=1, cols=2, horizontal_spacing=0.12)
                if d:
                    ue = np.asarray(d["unit_exhaust"], dtype=float)
                    sub.add_trace(go.Bar(x=["s0", "s1", "s2", "s3", "s4"], y=d["shell"], marker_color=C["s1"], name="shell"), 1, 1)
                    sub.add_trace(go.Bar(x=["D%d" % (k + 1) for k in range(10)], y=ue / max(float(ue.mean()), 1e-12),
                                         marker_color=C["s3"], name="divertor"), 1, 2)
                fig.data = []
                for tr in sub.data:
                    fig.add_trace(tr)
                fig.layout = sub.layout
                fig.update_layout(height=300, showlegend=False, bargap=0.12, **DARK)
                fig.update_xaxes(**AX)
                fig.update_yaxes(**AX)
                fig.layout.yaxis.title.text = "excess energy"
                fig.layout.yaxis2.title.text = "cumulative / mean"
            elif job == "manifold":
                self.dt_title.value = "<b>Bond expectations: PyQrack vs exact (current step)</b>"
                self.dt_html.value = ""
                fig.data = []
                if d and d.get("bond_exact"):
                    xe, yq = np.asarray(d["bond_exact"]), np.asarray(d["bond_qrack"])
                    lo, hi = float(min(xe.min(), yq.min())), float(max(xe.max(), yq.max()))
                    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", line=dict(color=C["axis"], dash="dash", width=1), hoverinfo="skip"))
                    fig.add_trace(go.Scatter(x=xe, y=yq, mode="markers", marker=dict(size=9, color=C["s2"], line=dict(color=C["surf"], width=1.5)),
                                             hovertemplate="exact %{x:.5f}<br>PyQrack %{y:.5f}<extra></extra>"))
                fig.layout = go.Layout(height=300, showlegend=False, **DARK)
                fig.update_xaxes(title_text="exact Majorana <sigma sigma>", **AX)
                fig.update_yaxes(title_text="PyQrack <sigma sigma>", **AX)
            else:
                self.dt_title.value = "<b>%s</b>" % {"criteria": "Last 7-criteria result", "coils": "Last coil point", "link": "Last link budget"}[job]
                fig.data = []
                fig.layout = go.Layout(height=10, xaxis=dict(visible=False, title_text=""),
                                       yaxis=dict(visible=False, title_text=""), **DARK)
                self.dt_box.layout.display = "none"
                last = [S["points"][i] for i in S["order"] if S["points"][i]["summary"] is not None]
                if not last:
                    self.dt_html.value = "<span style='color:%s'>Waiting for the first point</span>" % C["t3"]
                    return
                s_ = last[-1]["summary"]
                rows = [(LAB(k), FMT(v)) for k, v in s_.items() if not isinstance(v, (dict, list))]
                rows += [("%s (abs)" % k, FMT(v)) for k, v in (s_.get("raw") or {}).items()]
                self.dt_html.value = ("<div style='color:%s;font-size:11px'>%s</div><table>" % (C["t3"], self.coord_text(last[-1]["coords"]))
                                      + "".join("<tr><td style='color:%s'>%s</td><td>%s</td></tr>" % (C["t2"], k, v) for k, v in rows) + "</table>")

    def render_table(self):
        S = self.S
        if not S:
            self.table.value = "<span style='color:%s'>Every finished point is listed here.</span>" % C["t3"]
            self.table_hint.value = ""
            return
        keys = self.schema[S["job"]]["summary_metrics"]
        axk = [a["key"] for a in S["axes"]]
        order = S["order"][-300:]
        self.table_hint.value = "<span style='color:%s'>%s</span>" % (
            C["t3"], ("showing last 300 of %d" % len(S["order"])) if len(S["order"]) > 300 else "%d point(s)" % len(S["order"]))
        h = "<table><tr><th>#</th>" + "".join("<th>%s</th>" % LAB(k) for k in axk + keys) + "<th>s</th></tr>"
        for i in order[::-1]:
            P = S["points"][i]
            run = i == S["cur"] and S["running"] and P["summary"] is None
            st = " style='color:%s'" % C["s2"] if run else ""
            h += "<tr%s><td>%d</td>" % (st, i + 1) + "".join("<td>%s</td>" % FMT(P["coords"][k]) for k in axk)
            h += "".join("<td>%s</td>" % (FMT(P["summary"].get(k)) if P["summary"] else ("..." if run else "")) for k in keys)
            h += "<td>%s</td></tr>" % ("%.1f" % P["seconds"] if P.get("seconds") else "")
        self.table.value = h + "</table>"

    def close(self):
        self.alive = False
        self.r.stop()


In [ ]:
#@title 4. Open the dashboard
try:
    dash.close()            # re-running this cell replaces the previous dashboard
except NameError:
    pass
dash = StellaratorDashboard(machine, runner)
dash.show()


## Driving the dashboard from code

The widgets and code share one job runner, so a sweep started here also appears live in the dashboard above. `dash.wait()` blocks until it finishes; `runner.last_results` holds every point.


In [ ]:
# Example: a short NESS sweep over the divertor coupling, then a table of the results
dash.run_job({"job": "ness",
              "params": {"steps": 40, "field": "trimmed", "preset": "standard"},
              "sweep": {"key": "gamma_div", "min": 0.5, "max": 1.5, "n": 3, "scale": "lin"},
              "sweep2": None})
dash.wait()

import pandas as pd
pts = (runner.last_results or {}).get("points") or []
rows = [dict(p["coords"], **{k: v for k, v in p["summary"].items() if not isinstance(v, (list, dict))})
        for p in pts if p.get("summary")]
if rows:
    cols = ["gamma_div", "tau_E", "ex_in_ratio", "core_fraction", "per_p2m", "edge_rms_reduction_pct"]
    display(pd.DataFrame(rows)[[c for c in cols if c in rows[0]]])
else:
    print("No completed points (run stopped or failed; see Log in the dashboard above).")


## Notes

| Topic | Detail |
| :--- | :--- |
| Stop | Ends the run at the next step; the job and the widget updates each run in their own background thread, so the buttons stay responsive |
| Exports | **Export CSV** / **Export JSON** write the results next to the notebook and start a browser download in Colab |
| 3D view | Drag to rotate, scroll to zoom, hover sites for shell, module and block; the colour, shell, cutaway and layer controls redraw it |
| Re-running | Cell 4 can be re-run at any time; it closes the previous dashboard first |
| Same engine | Identical job definitions and numbers to `python3 stellarator_ui.py` (PySide6 desktop window) |
